# 02 · Training and evaluation sets

This notebook decides which recordings are used, and for what. It writes

- the table of PTB-XL with folds, labels and the subset for the comparison of adjacent segments (`PTBXL_SETS`);
- the results of the waveform check of the MIMIC-IV-ECG recordings that were examined (`MIMIC_CHECK`);
- the unlabelled training pool of MIMIC-IV-ECG (`POOL_RECORDS` and the array `MIMIC_POOL_ARRAY`);
- the nine evaluation sets in one table (`EVAL_RECORDS`) and the array of the MIMIC-IV-ECG evaluation recordings (`MIMIC_EVAL_ARRAY`);
- every number of the flow diagram (`SET_COUNTS`, Supplementary Fig. 1) and the characteristics of every set (`SET_CHARACTERISTICS`, Table 1 and Supplementary Table 1).

**Rules that hold for all sets**

1. *Exclusions before sampling.* Exclusions that are based on statements or on metadata (no diagnostic statement, statements of poor quality, pacing or premature beats, short exams, patients of the training pool)
   are applied first, and the sets are drawn from what remains.
2. *Waveform check when a candidate is examined.* Candidates are examined in random order, and a recording counts only if at least three R waves are found in each 5-s half of lead II (notebook 00).
   The examination stops when the set is complete, so the waveform check is applied only to the recordings that are examined.
3. *Patients.* The patients of the unlabelled training pool are in no evaluation set. Outside PTB-XL, an evaluation set has one recording or one pair of recordings per patient.
   The evaluation sets of one dataset are drawn independently of each other and may share patients; they are analysed separately.
4. *Seeds.* Every set has its own seed (`SET_SEED`), so that the same sets are drawn again.

The notebook needs the output of notebook 01. The first run takes about 6 min with 16 worker processes (waveform check of 39,694 recordings of MIMIC-IV-ECG, filtering of the 39,134 that are selected),
plus the time to download these recordings if they were not downloaded in notebook 01 (about 2 h). A later run takes about 2 min.
It can be interrupted and run again: the waveform checks are kept in `MIMIC_CHECK`, and arrays that exist are not written again.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import ast
import json

import pandas as pd

# ---- sizes of the sets and rules of selection --------------------------------------------------------------------------------
N_POOL_PATIENTS = 5000                  # patients of the unlabelled training pool (MIMIC-IV-ECG)
N_TEST = 10000                          # patients, with one recording each, in the test sets of MIMIC-IV-ECG, CODE-15% and EchoNext
N_PAIRS = 2000                          # pairs in a set of pairs (all candidates are taken when fewer exist)
PAIR_MIN_S, PAIR_MAX_S = 60, 86400      # pairs of recordings in MIMIC-IV-ECG: more than 1 min and at most 24 h apart
PTBXL_NOT_ADJACENT = {"PVC", "PAC", "PACE", "BIGU", "TRIGU"}       # PTB-XL statements (premature complexes, pacing, bigeminy, trigeminy) that exclude a test recording from the comparison of adjacent segments
SET_SEED = {"mimic_pool": 0x13528F5, "mimic_test": 0x13528FC, "mimic_pairs_all": 0x13528FD, "mimic_pairs_same_report": 0x13528FE,      # fixed numbers
            "code15_cd_change": 0x1352900, "code15_normal_change": 0x1352901, "code15_test": 0x1352902, "echonext_test": 0x135290B}

_needed = [PTBXL_DIR / "ptbxl_database.csv", PTBXL_ARRAY, MIMIC_DIR / "record_list.csv", MIMIC_DIR / "machine_measurements.csv", ECHONEXT_DIR / "echonext_metadata_100k.csv", ECHONEXT_ARRAY, ECHONEXT_INDEX]
_needed += [CODE15_CACHE / f"part{i}_{kind}" for i in range(18) for kind in ("filtered_f16.npy", "index.csv")]
_missing = [str(p.relative_to(MASTER)) for p in _needed if not p.exists()]
assert not _missing, f"output of notebook 01 is missing: {_missing[:5]}{' ...' if len(_missing) > 5 else ''}"
counts = {}                             # numbers for the flow diagram, collected section by section and written to SET_COUNTS at the end

## How a set is drawn

All sampled sets are drawn in the same way. The candidate patients, after the exclusions, are put into a random order with the seed of the set.
They are then examined one after another: a function `examine` looks at the recordings of the patient, asks for the waveform check of those it needs, and either selects a recording (or a pair) or rejects the patient.
This continues until the wanted number of patients has been selected. The numbers that are reported (examined, rejected) count the patients up to the one who completed the set.

`draw` examines the candidates in groups, only so that the waveform checks of a group can run in parallel; the result is the same as when the patients are examined strictly one by one,
because the outcome for a patient does not depend on any other patient.

In [ ]:
class WaveformCheck:
    """Results of the waveform check for the recordings of one dataset. results[key] = (readable 0/1, R waves in the first half, R waves in the second half, MD5 of the waveform file or "").
    compute(keys) checks recordings that have not been checked and returns their results as a dictionary."""

    def __init__(self, compute, results=None):
        self.compute = compute
        self.results = dict(results or {})

    def known(self, key):
        return int(key) in self.results

    def passes(self, key):
        readable, n_first, n_second, _ = self.results[int(key)]
        return readable == 1 and passes_waveform_check(n_first, n_second)

    def prepare(self, keys):
        new = [int(k) for k in dict.fromkeys(keys) if int(k) not in self.results]
        if new:
            self.results.update(self.compute(new))


def draw(candidates, n_target, examine, check, label=""):
    """Examines the candidates in the given order until n_target of them are selected (n_target = None: all candidates are examined).
    examine(patient, state) returns ("need", [recordings]) while it waits for waveform checks, and finally ("select", selection) or ("reject", reason).
    Returns the list [(patient, selection)] and the counts up to the patient who completed the set."""
    outcomes, position, n_selected, t0 = [], 0, 0, time.time()
    while position < len(candidates) and (n_target is None or n_selected < n_target):
        size = 500 if n_target is None else max(50, int(1.02 * (n_target - n_selected)) + 10)       # a little more than is still needed: almost all candidates pass
        group = candidates[position:position + size]; position += len(group)
        state = {p: {} for p in group}; done = {}
        while len(done) < len(group):
            needed = []
            for p in group:
                if p not in done:
                    kind, value = examine(p, state[p])
                    if kind == "need":
                        needed += value
                    else:
                        done[p] = (kind, value)
            check.prepare(needed)
        for p in group:
            outcomes.append((p, done[p][0], done[p][1], state[p])); n_selected += done[p][0] == "select"
    selected, summary = [], {"candidates": len(candidates), "examined": 0, "selected": 0, "rejected": {}, "recordings_failed_waveform_check": 0, "recordings_dropped_identical_file": 0}
    for p, kind, value, st in outcomes:
        summary["examined"] += 1
        summary["recordings_failed_waveform_check"] += len(st.get("failed", ())); summary["recordings_dropped_identical_file"] += len(st.get("identical", ()))
        if kind == "select":
            selected.append((p, value))
        else:
            summary["rejected"][value] = summary["rejected"].get(value, 0) + 1
        if n_target is not None and len(selected) == n_target:
            break
    summary["selected"] = len(selected)
    assert n_target is None or len(selected) == n_target, f"{label}: only {len(selected)} of {n_target} could be selected"
    print(f"{label}: {summary['selected']:,} selected of {summary['candidates']:,} candidates; {summary['examined']:,} examined, rejected {summary['rejected'] or 'none'} ({time.time() - t0:.0f} s)", flush=True)
    return selected, summary


def examine_single(order_of, check):
    """One recording per patient: the recordings of the patient are examined in the order order_of(patient), and the first one that passes the waveform check is selected."""
    def examine(patient, state):
        if "order" not in state:
            state["order"] = [int(r) for r in order_of(patient)]; state["k"] = 0; state["failed"] = set()
        while state["k"] < len(state["order"]):
            record = state["order"][state["k"]]
            if not check.known(record):
                return "need", [record]
            if check.passes(record):
                return "select", record
            state["failed"].add(record); state["k"] += 1
        return "reject", "waveform_check"
    return examine

## PTB-XL

**Labels.** A statement of PTB-XL is diagnostic when `scp_statements.csv` marks it so, and each diagnostic statement belongs to one of five superclasses
(NORM, MI, STTC, CD, HYP). A superclass is positive when any of its statements is present in the recording, whatever likelihood is attached to the statement.

**Sets.** Recordings without any diagnostic statement cannot be labelled and are excluded. The others are divided by the folds that the dataset recommends:
folds 1–8 for training, fold 9 for validation (selection of checkpoints and of decision thresholds), fold 10 for testing.

**Subset for the comparison of adjacent segments.** A decision change between the two 5-s halves of a recording should not be caused by an event that is present in one half only.
Test recordings with statements of premature complexes, bigeminy, trigeminy or pacing are therefore left out of this comparison, and so are recordings that fail the waveform check.
The AUROC is calculated on the whole test fold.

In [ ]:
ptb_db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv")
ptb_statements = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
ptb_diagnostic = ptb_statements[ptb_statements.diagnostic == 1]
ptb_codes = ptb_db.scp_codes.apply(ast.literal_eval)                                       # {statement: likelihood} for every recording

ptb = pd.DataFrame({"row": np.arange(len(ptb_db)), "ecg_id": ptb_db.ecg_id.values, "patient": ptb_db.patient_id.astype(int).values, "fold": ptb_db.strat_fold.values})
_has_diagnosis = ptb_codes.apply(lambda d: any(s in ptb_diagnostic.index for s in d)).values
ptb["set"] = np.where(~_has_diagnosis, "excluded", np.where(ptb.fold <= 8, "train", np.where(ptb.fold == 9, "val", "test")))

# subset of the test fold for the comparison of adjacent segments
_is_test = (ptb.set == "test").values
_not_adjacent = ptb_codes.apply(lambda d: any(s in PTBXL_NOT_ADJACENT for s in d)).values
_ptb_array = np.load(PTBXL_ARRAY, mmap_mode="r")
_passes = np.zeros(len(ptb), bool)
for _i in np.where(_is_test)[0]:                                                           # R waves of lead II (index 1) of the cached recording; about 1 ms per recording
    _passes[_i] = passes_waveform_check(*count_r_waves(_ptb_array[_i, 1]))
ptb["adjacent_ok"] = (_is_test & ~_not_adjacent & _passes).astype(int)

for _c in CLASSES:
    _members = set(ptb_diagnostic.index[ptb_diagnostic.diagnostic_class == _c])
    ptb[_c] = ptb_codes.apply(lambda d: int(any(s in _members for s in d))).values
ptb.to_csv(PTBXL_SETS, index=False)

counts["ptbxl"] = {"recordings": len(ptb), "patients": int(ptb.patient.nunique()), "excluded_no_diagnostic_statement": int((ptb.set == "excluded").sum()),
                   "recordings_with_diagnostic_statement": int((ptb.set != "excluded").sum()), "train": int((ptb.set == "train").sum()), "val": int((ptb.set == "val").sum()), "test": int(_is_test.sum()),
                   "test_excluded_by_statement": int((_is_test & _not_adjacent).sum()), "test_failed_waveform_check": int((_is_test & ~_not_adjacent & ~_passes).sum()), "test_adjacent": int(ptb.adjacent_ok.sum())}
print(counts["ptbxl"])

## MIMIC-IV-ECG

### Machine statements: exclusions and labels

`machine_measurements.csv` holds up to 18 lines of interpretation statements for every recording (`report_0` … `report_17`). They are used for three things.

**Exclusions.** A recording is excluded when its statements mention poor data quality, artefact, baseline wander, muscle tremor or reversed leads, warn that the interpretation may be affected,
or mention pacing or premature beats. These are the conditions under which the two halves of a recording, or two recordings, are expected to differ for reasons other than the reading of the model.

**Reference labels for the AUROC.** The five superclasses are defined operationally by terms in the statements, including statements that are qualified as possible or probable:

| Class | Terms |
|---|---|
| MI | infarct, myocardial infarction |
| STTC | ST-T, T wave, ST depression, ST elevation or any other statement with the word ST, ischaemia, repolarization (statement lines about early repolarization are ignored), prolonged QT, digitalis |
| CD | bundle branch block, fascicular block, atrioventricular block, intraventricular conduction delay, pre-excitation (and their usual abbreviations) |
| HYP | ventricular hypertrophy, atrial enlargement, atrial abnormality, atrial overload |
| NORM | the summary statement "normal ECG" (not "abnormal ECG", not "otherwise normal ECG") and none of the four classes above |

These labels are not confirmed by a physician, so the AUROC in this dataset is approximate.

**Identical reports.** Two recordings have identical machine reports when they have exactly the same set of statement lines (after removal of surrounding blanks and of differences in case).

The patterns are regular expressions that are applied to the statement lines in lower case, joined into one text per recording.

In [ ]:
EXCLUSION_PATTERNS = {"poor_quality": r"poor data quality|artifact|baseline wander|muscle tremor|leads? reversed|interpretation may be",
                      "pacing": r"paced|pacemaker",
                      "ventricular_premature_beats": r"ventricular premature|premature ventricular|pvc",
                      "atrial_premature_beats": r"atrial premature|premature atrial|pac\b|supraventricular premature"}
LABEL_PATTERNS = {"MI": r"infarct|myocardial infarction",
                  "STTC": r"st-t|t wave|st depression|st elevation|\bst\b|ischemi|repolarization|prolonged qt|qt interval prolong|long qt|digitalis",
                  "CD": r"bundle branch block|\brbbb\b|\blbbb\b|fascicular block|\blafb\b|\blpfb\b|a-?v block|intraventricular conduction|\bivcd\b|wolff|wpw|pre-?excitation",
                  "HYP": r"ventricular hypertrophy|\blvh\b|\brvh\b|atrial enlargement|atrial abnormality|atrial overload|biatrial"}

_report_columns = [f"report_{i}" for i in range(18)]
mm = pd.read_csv(MIMIC_DIR / "machine_measurements.csv", usecols=["subject_id", "study_id", "ecg_time"] + _report_columns, low_memory=False)
_lines = mm[_report_columns].fillna("").astype(str)
_text = _lines.agg(" | ".join, axis=1).str.lower()                                         # all statement lines of a recording as one text
_lines = _lines.apply(lambda column: column.str.lower().str.strip())
_text_stt = _lines.apply(lambda column: column.where(~column.str.contains("early repol", regex=False), "")).agg(" | ".join, axis=1)    # the same without the lines about early repolarization

_excluded_for = {name: _text.str.contains(pattern, regex=True).values for name, pattern in EXCLUSION_PATTERNS.items()}
mm["excluded"] = np.any(list(_excluded_for.values()), axis=0)
for _c in ("MI", "STTC", "CD", "HYP"):
    mm[_c] = (_text_stt if _c == "STTC" else _text).str.contains(LABEL_PATTERNS[_c], regex=True).values.astype(int)
mm["NORM"] = (_text.str.contains(r"(?<![a-z])normal ecg", regex=True) & ~_text.str.contains("otherwise normal", regex=True)).values.astype(int) * (1 - mm[["MI", "STTC", "CD", "HYP"]].max(axis=1).values)
mm["report"] = [frozenset(line for line in row if line) for row in _lines.values.tolist()]   # the set of statement lines, for the pairs with identical reports

print(f"recordings {len(mm):,} | excluded by statement {int(mm.excluded.sum()):,} (" + ", ".join(f"{k} {int(v.sum()):,}" for k, v in _excluded_for.items()) + ")")
print("labels from machine statements, % of all recordings: " + ", ".join(f"{c} {100 * mm[c].mean():.1f}" for c in CLASSES))

### Times of the recordings

The time of a recording is given in `record_list.csv` and again in `machine_measurements.csv`. For 17,850 recordings the two times differ.
The time of `record_list.csv` is used to order the recordings of a patient. A recording whose two times differ is not used in any pair, because the interval between the recordings of a pair must be known;
such recordings remain candidates for the training pool and for the test set, where the time has no role.

In [ ]:
_record_list = pd.read_csv(MIMIC_DIR / "record_list.csv", usecols=["study_id", "ecg_time", "path"]).rename(columns={"ecg_time": "time_record_list"})
mimic = mm.merge(_record_list, on="study_id")
mimic["time"] = pd.to_datetime(mimic.time_record_list)
mimic["time_consistent"] = (mimic.time == pd.to_datetime(mimic.ecg_time)).values
mimic = mimic.sort_values(["subject_id", "time", "study_id"]).reset_index(drop=True)          # recordings of a patient in the order of time

MIMIC_PATH = dict(zip(mimic.study_id.values, mimic.path.values))
MIMIC_TIME = dict(zip(mimic.study_id.values, mimic.time.values.astype("datetime64[s]").astype(np.int64)))      # seconds
MIMIC_REPORT = dict(zip(mimic.study_id.values, mimic.report.values))
_clean = mimic[~mimic.excluded]
clean_records = {p: v.values for p, v in _clean.groupby("subject_id").study_id}                                # recordings that pass the statement rules, by patient, in time order
pair_records = {p: v.values for p, v in _clean[_clean.time_consistent].groupby("subject_id").study_id}         # the same, without recordings whose two times differ
clean_patients = np.array(sorted(clean_records))

counts["mimic"] = {"recordings": len(mimic), "patients": int(mimic.subject_id.nunique()), "excluded_by_statement": int(mimic.excluded.sum()),
                   "excluded_by_statement_reasons": {k: int(v.sum()) for k, v in _excluded_for.items()},
                   "recordings_after_statement_exclusion": len(_clean), "patients_after_statement_exclusion": len(clean_patients),
                   "recordings_time_mismatch": int((~mimic.time_consistent).sum())}
print({k: v for k, v in counts["mimic"].items() if not isinstance(v, dict)})

### Waveform check of MIMIC-IV-ECG recordings

A recording of MIMIC-IV-ECG is read from its WFDB files, filtered and checked when it is first examined (`check_mimic_record` in notebook 00).
The result is appended to `MIMIC_CHECK`, so that no recording is checked twice, also when the notebook is run again.
Files that are not on disk are downloaded first; a download that fails stops the notebook instead of being counted as a failed check.
A check takes about 30 ms in one process, most of it for reading the files; with 16 worker processes we measured about 7 ms per recording.

In [ ]:
CHECK_COLUMNS = ["record", "readable", "n_r_first", "n_r_second", "md5"]


def compute_mimic_check(records):
    """Waveform check of MIMIC-IV-ECG recordings that have not been checked before. Results are appended to MIMIC_CHECK after every 2,000 recordings."""
    download_mimic_records([MIMIC_PATH[s] for s in records])
    out, t0 = {}, time.time()
    for start in range(0, len(records), 2000):
        part = records[start:start + 2000]
        results = list(parallel_map(check_mimic_record, [MIMIC_PATH[s] for s in part], n_jobs=None if len(part) >= 64 else 1))
        pd.DataFrame([(s, *r) for s, r in zip(part, results)], columns=CHECK_COLUMNS).to_csv(MIMIC_CHECK, mode="a", header=not MIMIC_CHECK.exists(), index=False)
        out.update(zip(part, results))
        if len(records) > 2000:
            print(f"   waveform check: {min(start + 2000, len(records)):,} of {len(records):,} recordings, {(time.time() - t0) / 60:.1f} min", flush=True)
    return out


_stored = pd.read_csv(MIMIC_CHECK).fillna({"md5": ""}) if MIMIC_CHECK.exists() else pd.DataFrame(columns=CHECK_COLUMNS)
mimic_check = WaveformCheck(compute_mimic_check, {int(r.record): (int(r.readable), int(r.n_r_first), int(r.n_r_second), str(r.md5)) for r in _stored.itertuples()})
print(f"waveform checks already stored: {len(mimic_check.results):,}")

### Unlabelled training pool

The pool provides ECGs without labels for distillation (notebook 03). 5,000 patients are drawn at random from the patients who have at least one recording after the statement exclusions,
and every such recording of these patients that passes the waveform check enters the pool. A few of the patients have no recording that passes, so the pool has slightly fewer than 5,000 patients.
The labels of these recordings are never used.

In [ ]:
_rng = np.random.default_rng(SET_SEED["mimic_pool"])
pool_patients = [int(p) for p in _rng.permutation(clean_patients)[:N_POOL_PATIENTS]]
_pool_candidates = [int(s) for p in pool_patients for s in clean_records[p]]
mimic_check.prepare(_pool_candidates)                                                       # first run: 21,444 recordings, about 2.5 min with 16 worker processes

pool = pd.DataFrame([(p, s) for p in pool_patients for s in clean_records[p] if mimic_check.passes(s)], columns=["patient", "record"]).astype(int)
pool["row"] = np.arange(len(pool))                                                          # row of MIMIC_POOL_ARRAY
pool.to_csv(POOL_RECORDS, index=False)
counts["mimic"]["pool"] = {"patients_drawn": len(pool_patients), "recordings_examined": len(_pool_candidates), "recordings_failed_waveform_check": len(_pool_candidates) - len(pool),
                           "recordings": len(pool), "patients": int(pool.patient.nunique())}
print(counts["mimic"]["pool"])

### Test set

All patients outside the training pool are candidates. They are put into a random order, the recordings of each patient are put into a random order as well,
and the first recording of the patient that passes the waveform check is taken, until 10,000 patients have a recording.
This set is used for the decision change between adjacent segments and, with the labels from the machine statements, for the AUROC.

In [ ]:
_pool_set = set(pool_patients)
eval_patients = np.array([p for p in clean_patients if int(p) not in _pool_set])
counts["mimic"]["evaluation_candidate_patients"] = len(eval_patients)

_rng = np.random.default_rng(SET_SEED["mimic_test"])
_order = [int(p) for p in _rng.permutation(eval_patients)]
_record_order = {p: _rng.permutation(clean_records[p]) for p in _order}                        # random order of the recordings of each patient, drawn with the same generator in the order of the list
mimic_test, _summary = draw(_order, N_TEST, examine_single(lambda p: _record_order[p], mimic_check), mimic_check, "mimic_test")
counts["mimic"]["sets"] = {"mimic_test": _summary}

### Pairs of recordings

A pair consists of two successive recordings of one patient that were taken more than 1 min and at most 24 h apart. Successive means that no other usable recording of the patient lies between them;
usable recordings are those that pass the statement rules, have consistent times and have not been dropped by the waveform check.
For each patient the pair with the shortest interval is taken. If one of its recordings fails the waveform check, that recording is dropped and the pairs of the patient are formed again.
If the two waveform files of a pair are identical (the same recording stored twice; compared by their MD5 checksums), the second recording is dropped in the same way.

Two sets of 2,000 pairs are drawn independently:

- **all pairs** (`mimic_pairs_all`): no further condition;
- **pairs with identical machine reports** (`mimic_pairs_same_report`): the two recordings have exactly the same set of statement lines, so no change is documented between them.
  These pairs approximate a repeated recording without a change in the patient and are the main comparison between recordings.

In [ ]:
def successive_pairs(records, dropped=(), same_report=False):
    """Pairs of successive recordings (given in time order) more than 1 min and at most 24 h apart, as (interval in seconds, first, second), sorted by the interval.
    Recordings in `dropped` are removed before the pairs are formed."""
    left = [int(s) for s in records if int(s) not in dropped]; out = []
    for a, b in zip(left[:-1], left[1:]):
        interval = float(MIMIC_TIME[b] - MIMIC_TIME[a])
        if PAIR_MIN_S < interval <= PAIR_MAX_S and (not same_report or MIMIC_REPORT[a] == MIMIC_REPORT[b]):
            out.append((interval, a, b))
    return sorted(out)


def examine_pair(same_report):
    def examine(patient, state):
        dropped = state.setdefault("dropped", set()); failed = state.setdefault("failed", set()); identical = state.setdefault("identical", set())
        while True:
            pairs = successive_pairs(pair_records.get(patient, ()), dropped, same_report)
            if not pairs:
                return "reject", "no_pair" if not dropped else "waveform_check" if not identical else "identical_file" if not failed else "waveform_check_and_identical_file"
            interval, a, b = pairs[0]
            unknown = [s for s in (a, b) if not mimic_check.known(s)]
            if unknown:
                return "need", unknown
            bad = [s for s in (a, b) if not mimic_check.passes(s)]
            if bad:
                dropped.update(bad); failed.update(bad)
                continue
            if mimic_check.results[a][3] and mimic_check.results[a][3] == mimic_check.results[b][3]:      # identical waveform files
                dropped.add(b); identical.add(b)
                continue
            return "select", (a, b, interval)
    return examine


mimic_pairs = {}
for _name, _same in (("mimic_pairs_all", False), ("mimic_pairs_same_report", True)):
    _candidates = np.array([int(p) for p in eval_patients if p in pair_records and successive_pairs(pair_records[p], (), _same)])
    _order = [int(p) for p in np.random.default_rng(SET_SEED[_name]).permutation(_candidates)]
    mimic_pairs[_name], counts["mimic"]["sets"][_name] = draw(_order, N_PAIRS, examine_pair(_same), mimic_check, _name)

# numbers for the flow diagram: candidates for pairs before the recordings with inconsistent times are removed, and the number of such recordings among the candidates for evaluation
counts["mimic"]["pair_candidate_patients_before_time_exclusion"] = int(sum(1 for p in eval_patients if successive_pairs(clean_records[p])))
counts["mimic"]["recordings_excluded_from_pairs_time_mismatch"] = int((~_clean.time_consistent & ~_clean.subject_id.isin(_pool_set)).sum())
print({k: v for k, v in counts["mimic"].items() if isinstance(v, int)})

### Table and arrays of the MIMIC-IV-ECG recordings

The recordings of the three evaluation sets are stored once in `MIMIC_EVAL_ARRAY`, in the order in which they were selected (test set first); a recording that belongs to two sets is stored once.
The recordings of the pool are stored in `MIMIC_POOL_ARRAY` in the order of `POOL_RECORDS`.
Filtering takes about 2 min for the two arrays with 16 worker processes (39,134 recordings).

In [ ]:
_rows = [{"dataset": "mimic", "set": "mimic_test", "patient": p, "record": s, "pair_id": -1, "role": "single", "interval_sec": np.nan, "adjacent_ok": 1} for p, s in mimic_test]
for _name, _selected in mimic_pairs.items():
    for _k, (p, (a, b, interval)) in enumerate(_selected):
        _rows += [{"dataset": "mimic", "set": _name, "patient": p, "record": s, "pair_id": _k, "role": role, "interval_sec": interval, "adjacent_ok": 0} for role, s in (("a", a), ("b", b))]
mimic_eval = pd.DataFrame(_rows)
mimic_eval_records = list(dict.fromkeys(int(s) for s in mimic_eval.record))                   # each recording once, in the order of selection
mimic_eval["array"] = "mimic_eval"
mimic_eval["row"] = mimic_eval.record.map({s: i for i, s in enumerate(mimic_eval_records)})   # row of MIMIC_EVAL_ARRAY
_labels = mimic.set_index("study_id")[CLASSES]
for _c in CLASSES:
    mimic_eval[_c] = _labels[_c].reindex(mimic_eval.record.values).values


def build_mimic_array(path, records):
    """Filters the given recordings of MIMIC-IV-ECG and writes them to `path`, unless an array with this number of recordings exists."""
    if path.exists() and np.load(path, mmap_mode="r").shape[0] == len(records):
        print(f"{path.name}: present ({len(records):,} recordings)")       # delete the file to write it again
        return
    download_mimic_records([MIMIC_PATH[s] for s in records])
    write_array(path, len(records), parallel_map(filter_mimic_record, [MIMIC_PATH[s] for s in records]), report_every=5000)
    print(f"{path.name}: written ({len(records):,} recordings)")


build_mimic_array(MIMIC_POOL_ARRAY, pool.record.tolist())
build_mimic_array(MIMIC_EVAL_ARRAY, mimic_eval_records)

## CODE-15%

**Exams that are used.** Only complete 10-s exams are used: cached exams whose signal spans at least 3,868 of the 4,000 samples of the 10-s window (column `signal_samples` of the index written by notebook 01).
All other exams, among them the many exams of about 7 s, are excluded before sampling.

**Labels.** Conduction disturbance (CD) is any of first-degree atrioventricular block, right bundle branch block and left bundle branch block; the flag for a normal ECG is used as provided (NORM).
The dataset has no labels that correspond to the other three classes.

**Test set.** 10,000 patients with one exam each: the patients are taken in random order, the exams of each patient in random order, and the first exam that passes the waveform check is used.

**Pairs with a change of label.** These sets ask whether a model follows a change that is documented by the reference: for a patient who has exams with and without the label,
one exam with the label and one exam without it are drawn at random among the exams that pass the waveform check. The dataset gives no dates, so the interval between the two exams is not restricted.
All 1,732 candidate patients are examined for conduction disturbance; for the normal flag, 2,000 pairs are drawn.

The waveform check reads lead II from the cached arrays (about 1 ms per exam; less than 1 min for the three sets).

In [ ]:
code = pd.concat([pd.read_csv(CODE15_CACHE / f"part{i}_index.csv").assign(part=i) for i in range(18)], ignore_index=True)
code["CD"] = (code["1dAVb"] | code.RBBB | code.LBBB).astype(int)
code["NORM"] = code.normal_ecg.astype(int)
_complete = code[code.signal_samples >= CODE15_MIN_SAMPLES]
_all_exams = pd.read_csv(CODE15_DIR / "exams.csv", usecols=["exam_id", "patient_id"])
counts["code15"] = {"exams": len(_all_exams), "patients": int(_all_exams.patient_id.nunique()), "exams_cached": len(code), "excluded_shorter_than_10_s": len(_all_exams) - len(_complete),
                    "exams_after_exclusion": len(_complete), "patients_after_exclusion": int(_complete.patient_id.nunique()), "sets": {}}
print({k: v for k, v in counts["code15"].items() if k != "sets"})

code_exams = {p: v.values for p, v in _complete.groupby("patient_id").exam_id}               # complete exams of each patient, in the order of the cache
CODE_ARRAY = {e: f"code15/part{i}" for e, i in zip(_complete.exam_id.values, _complete.part.values)}
CODE_ROW = dict(zip(_complete.exam_id.values, _complete.row.values))
CODE_LABEL = {c: dict(zip(_complete.exam_id.values, _complete[c].values)) for c in ("CD", "NORM")}
_code_arrays = {i: np.load(CODE15_CACHE / f"part{i}_filtered_f16.npy", mmap_mode="r") for i in range(18)}
_code_place = lambda e: (int(CODE_ARRAY[e].split("part")[1]), int(CODE_ROW[e]))          # (part, row) of an exam
code_check = WaveformCheck(lambda exams: {e: (1, *count_r_waves(_code_arrays[_code_place(e)[0]][_code_place(e)[1], 1]), "") for e in sorted(exams, key=_code_place)})   # sorted: the arrays are read front to back


def examine_change(exams_of, label, check, seed):
    """Pair with a change of label: all exams of the patient are checked; one exam with the label and one without it are drawn at random among those that pass (generator seeded with the seed of the set and the patient)."""
    def examine(patient, state):
        exams = [int(e) for e in exams_of[patient]]
        unknown = [e for e in exams if not check.known(e)]
        if unknown:
            return "need", unknown
        state["failed"] = {e for e in exams if not check.passes(e)}
        with_label = [e for e in exams if label[e] == 1 and check.passes(e)]; without_label = [e for e in exams if label[e] == 0 and check.passes(e)]
        if not with_label or not without_label:
            return "reject", "waveform_check"
        rng = np.random.default_rng([seed, int(patient)])
        return "select", (int(rng.choice(with_label)), int(rng.choice(without_label)))
    return examine


code_pairs = {}
for _name, _c, _n in (("code15_cd_change", "CD", None), ("code15_normal_change", "NORM", N_PAIRS)):
    _candidates = np.array(sorted(p for p, exams in code_exams.items() if len({CODE_LABEL[_c][e] for e in exams}) > 1))      # patients with exams with and without the label
    _order = [int(p) for p in np.random.default_rng(SET_SEED[_name]).permutation(_candidates)]
    code_pairs[_name], counts["code15"]["sets"][_name] = draw(_order, _n, examine_change(code_exams, CODE_LABEL[_c], code_check, SET_SEED[_name]), code_check, _name)

_order = [int(p) for p in np.random.default_rng(SET_SEED["code15_test"]).permutation(np.array(sorted(code_exams)))]
code_test, counts["code15"]["sets"]["code15_test"] = draw(
    _order, N_TEST, examine_single(lambda p: np.random.default_rng([SET_SEED["code15_test"], int(p)]).permutation(code_exams[p]), code_check), code_check, "code15_test")

## EchoNext

All 100,000 ECGs are candidates; the dataset has no statements on which an exclusion could be based. The test set has 10,000 patients with one ECG each, drawn as in the other datasets.
The reference is the flag for a left ventricular wall thickness of 13 mm or more on echocardiography (`lvwt_gte_13_flag`), stored in the column HYP of the table of evaluation recordings,
because it is compared with the hypertrophy output of the models. An ECG is identified by its row in the cached array.

In [ ]:
_echo_meta = pd.read_csv(ECHONEXT_DIR / "echonext_metadata_100k.csv", index_col=0)
echo = _echo_meta.iloc[pd.read_csv(ECHONEXT_INDEX).metadata_row.values].reset_index(drop=True)      # the metadata in the order of the rows of the cached array
echo["row"] = np.arange(len(echo))
echo_ecgs = {p: v.values for p, v in echo.groupby("patient_key").row}
ECHO_FLAG = echo.lvwt_gte_13_flag.astype(int).values
_echo_array = np.load(ECHONEXT_ARRAY, mmap_mode="r")
echo_check = WaveformCheck(lambda rows: {r: (1, *count_r_waves(_echo_array[r, 1]), "") for r in sorted(rows)})
counts["echonext"] = {"recordings": len(echo), "patients": len(echo_ecgs), "sets": {}}

_order = [int(p) for p in np.random.default_rng(SET_SEED["echonext_test"]).permutation(np.array(sorted(echo_ecgs)))]
echo_test, counts["echonext"]["sets"]["echonext_test"] = draw(
    _order, N_TEST, examine_single(lambda p: np.random.default_rng([SET_SEED["echonext_test"], int(p)]).permutation(echo_ecgs[p]), echo_check), echo_check, "echonext_test")

## Table of the evaluation recordings

All evaluation sets are collected in one table with one line per recording and set. A recording that belongs to two sets has two lines.
**The order of the lines is the order of the rows of every file of model outputs** (notebook 04), so it must not be changed once outputs have been computed.
The lines are ordered by array and by row within the array (CODE-15% part by part, EchoNext, PTB-XL), followed by the MIMIC-IV-ECG recordings in the order of `MIMIC_EVAL_ARRAY`;
reading the arrays in this order is sequential on disk. The order has no meaning for the analysis.

The column `adjacent_ok` marks the recordings that are used for the comparison of adjacent segments: the subset of the PTB-XL test fold and the test sets of the other datasets.
Reference labels are given where the dataset has them (PTB-XL and MIMIC-IV-ECG: five classes; CODE-15%: NORM and CD; EchoNext: HYP = wall thickness of 13 mm or more).

In [ ]:
_rows = []
for _set, _name in (("val", "ptbxl_val"), ("test", "ptbxl_test")):
    for r in ptb[ptb.set == _set].itertuples():
        _rows.append({"dataset": "ptbxl", "set": _name, "patient": r.patient, "record": r.ecg_id, "pair_id": -1, "role": "single", "array": "ptbxl", "row": r.row, "interval_sec": np.nan,
                      "adjacent_ok": r.adjacent_ok if _set == "test" else 0, **{c: getattr(r, c) for c in CLASSES}})
for _name, _selected in code_pairs.items():
    for _k, (p, (e_with, e_without)) in enumerate(_selected):
        for role, e in (("pos", e_with), ("neg", e_without)):
            _rows.append({"dataset": "code15", "set": _name, "patient": p, "record": e, "pair_id": _k, "role": role, "array": CODE_ARRAY[e], "row": CODE_ROW[e], "interval_sec": np.nan,
                          "adjacent_ok": 0, "NORM": CODE_LABEL["NORM"][e], "CD": CODE_LABEL["CD"][e]})
for p, e in code_test:
    _rows.append({"dataset": "code15", "set": "code15_test", "patient": p, "record": e, "pair_id": -1, "role": "single", "array": CODE_ARRAY[e], "row": CODE_ROW[e], "interval_sec": np.nan,
                  "adjacent_ok": 1, "NORM": CODE_LABEL["NORM"][e], "CD": CODE_LABEL["CD"][e]})
for p, r in echo_test:
    _rows.append({"dataset": "echonext", "set": "echonext_test", "patient": p, "record": r, "pair_id": -1, "role": "single", "array": "echonext", "row": r, "interval_sec": np.nan,
                  "adjacent_ok": 1, "HYP": ECHO_FLAG[r]})

EVAL_COLUMNS = ["dataset", "set", "patient", "record", "pair_id", "role", "array", "row", "interval_sec", "adjacent_ok"] + CLASSES
_others = pd.DataFrame(_rows).sort_values(["array", "row"], kind="stable")                    # stable: lines of one recording keep the order in which the sets were added
_mimic = mimic_eval.sort_values(["row", "set"], kind="stable")
eval_records = pd.concat([_others[EVAL_COLUMNS], _mimic[EVAL_COLUMNS]], ignore_index=True)
eval_records = eval_records.astype({"patient": "int64", "record": "int64", "pair_id": "int64", "row": "int64", "adjacent_ok": "int64", **{c: "float64" for c in CLASSES}})
eval_records.to_csv(EVAL_RECORDS, index=False)
print(eval_records.groupby(["dataset", "set"], sort=False).size().to_string())
print("lines:", len(eval_records))

### Checks of the sets

The cell verifies the properties that the analysis relies on.

In [ ]:
assert sorted(eval_records.set.unique()) == sorted(EVAL_SETS)
_pool_patients = set(pool.patient)
assert not (_pool_patients & set(eval_records.patient[eval_records.dataset == "mimic"])), "a patient of the training pool is in an evaluation set"
for _name in EVAL_SETS:
    q = eval_records[eval_records.set == _name]
    if q.pair_id.max() >= 0:                                                                  # sets of pairs: two recordings of one patient, one pair per patient
        assert (q.groupby("pair_id").size() == 2).all() and (q.groupby("pair_id").patient.nunique() == 1).all() and q.patient.nunique() == len(q) // 2 and q.record.is_unique, _name
        assert set(q.role) in ({"a", "b"}, {"pos", "neg"}), _name
    elif not _name.startswith("ptbxl"):                                                       # test sets outside PTB-XL: one recording per patient
        assert q.patient.is_unique, _name
for _name in ("mimic_pairs_all", "mimic_pairs_same_report"):
    q = eval_records[(eval_records.set == _name) & (eval_records.role == "a")]
    assert ((q.interval_sec > PAIR_MIN_S) & (q.interval_sec <= PAIR_MAX_S)).all(), _name
for _name, _c in (("code15_cd_change", "CD"), ("code15_normal_change", "NORM")):                # the label is present in the recording with the role "pos" only
    q = eval_records[eval_records.set == _name]
    assert (q[_c][q.role == "pos"] == 1).all() and (q[_c][q.role == "neg"] == 0).all(), _name
assert np.load(MIMIC_POOL_ARRAY, mmap_mode="r").shape[0] == len(pool) and np.load(MIMIC_EVAL_ARRAY, mmap_mode="r").shape[0] == len(mimic_eval_records)
print("all checks passed")

## Numbers of the flow diagram (Supplementary Fig. 1)

`SET_COUNTS` holds every number of the flow diagram: recordings and patients of each dataset, exclusions before sampling, and, for each sampled set,
the candidates, the patients examined, selected and rejected (with the reason), and the recordings that failed the waveform check or were dropped because of an identical file.

In [ ]:
SET_COUNTS.write_text(json.dumps(counts, indent=1), encoding="utf-8")
_flow = []
for _dataset in ("mimic", "code15", "echonext"):
    for _name, s in counts[_dataset]["sets"].items():
        _flow.append({"set": _name, "candidates": s["candidates"], "examined": s["examined"], "selected": s["selected"], "rejected": sum(s["rejected"].values()),
                      "recordings_failed_waveform_check": s["recordings_failed_waveform_check"], "recordings_dropped_identical_file": s["recordings_dropped_identical_file"]})
print(pd.DataFrame(_flow).to_string(index=False))

## Characteristics of the sets (Table 1, Supplementary Table 1)

For every set: numbers of recordings, patients and pairs; age (median and quartiles) and percentage of women, both summarized over recordings; the prevalence of the reference labels; and, for the pairs of MIMIC-IV-ECG, the interval between the two recordings.

- Ages above 89 years are set to 90 before the summary, because the datasets do not give them exactly (PTB-XL records them as 300, MIMIC-IV gives an anchor age of 91, EchoNext caps the age at 90).
- MIMIC-IV-ECG has neither age nor sex. If the patients table of MIMIC-IV is present (optional, notebook 01), the sex is taken from it and the age at the recording is calculated as
  anchor age + (year of the recording − anchor year). Otherwise these fields are left empty.
- In PTB-XL, `sex` = 1 denotes women. In CODE-15%, women are the exams with `is_male` false.

In [ ]:
def describe_set(name, dataset, recordings, patients, pairs=None, age=None, women=None, labels=None, interval_h=None):
    """One line of the table of characteristics. age and women: one value per recording (women 1/0); labels: table with one column per class; interval_h: one value per pair."""
    line = {"set": name, "dataset": dataset, "recordings": int(recordings), "patients": int(patients), "pairs": np.nan if pairs is None else int(pairs)}
    a = np.asarray([] if age is None else age, dtype=float); a = np.minimum(a[np.isfinite(a)], 90)
    line.update({"age_n": len(a), "age_median": np.median(a) if len(a) else np.nan, "age_q1": np.percentile(a, 25) if len(a) else np.nan, "age_q3": np.percentile(a, 75) if len(a) else np.nan})
    w = np.asarray([] if women is None else women, dtype=float); w = w[np.isfinite(w)]
    line.update({"sex_n": len(w), "women_n": int(w.sum()), "women_pct": 100 * w.mean() if len(w) else np.nan})
    for c in CLASSES:
        line[f"prevalence_{c}"] = 100 * float(np.mean(labels[c])) if labels is not None and c in labels else np.nan
    g = np.asarray([] if interval_h is None else interval_h, dtype=float)
    line.update({"interval_h_median": np.median(g) if len(g) else np.nan, "interval_h_q1": np.percentile(g, 25) if len(g) else np.nan, "interval_h_q3": np.percentile(g, 75) if len(g) else np.nan,
                 "interval_within_1h_n": int((g <= 1).sum()) if len(g) else np.nan, "interval_within_1h_pct": 100 * float((g <= 1).mean()) if len(g) else np.nan})
    return line


_lines = []
# PTB-XL
for _name, _mask in (("ptbxl_train", ptb.set == "train"), ("ptbxl_val", ptb.set == "val"), ("ptbxl_test", ptb.set == "test"), ("ptbxl_test_adjacent", ptb.adjacent_ok == 1)):
    q = ptb[_mask.values]
    _lines.append(describe_set(_name, "ptbxl", len(q), q.patient.nunique(), age=ptb_db.age.values[q.row.values], women=ptb_db.sex.values[q.row.values], labels=q[CLASSES]))

# MIMIC-IV-ECG: age and sex from the patients table of MIMIC-IV, if it is present
if MIMIC_PATIENTS.exists():
    _patients = pd.read_csv(MIMIC_PATIENTS, usecols=["subject_id", "gender", "anchor_age", "anchor_year"]).drop_duplicates("subject_id").set_index("subject_id")
    _year = mimic.set_index("study_id").time.dt.year
else:
    _patients = None
    print("patients table of MIMIC-IV not found: age and sex of the MIMIC-IV-ECG sets are left empty")


def mimic_age_and_sex(table):
    """Age at the recording and sex (1 = woman) for the recordings of a table with the columns patient and record; (None, None) without the patients table."""
    if _patients is None:
        return None, None
    p = _patients.reindex(table.patient.values)
    age = p.anchor_age.values + (_year.reindex(table.record.values).values - p.anchor_year.values)
    return age, np.where(p.gender.isna().values, np.nan, (p.gender.astype(str).str.upper() == "F").values.astype(float))


_age, _women = mimic_age_and_sex(pool)
_lines.append(describe_set("mimic_pool", "mimic", len(pool), pool.patient.nunique(), age=_age, women=_women, labels=_labels.reindex(pool.record.values)))
for _name in ("mimic_test", "mimic_pairs_all", "mimic_pairs_same_report"):
    q = eval_records[eval_records.set == _name]; _age, _women = mimic_age_and_sex(q); _is_pair = _name != "mimic_test"
    _lines.append(describe_set(_name, "mimic", len(q), q.patient.nunique(), pairs=len(q) // 2 if _is_pair else None, age=_age, women=_women, labels=None if _is_pair else q[CLASSES],
                               interval_h=q.interval_sec[q.role == "a"].values / 3600 if _is_pair else None))

# CODE-15%
_code = code.set_index("exam_id")
for _name in ("code15_test", "code15_cd_change", "code15_normal_change"):
    q = eval_records[eval_records.set == _name]; e = _code.loc[q.record.values]; _is_pair = _name != "code15_test"
    _lines.append(describe_set(_name, "code15", len(q), q.patient.nunique(), pairs=len(q) // 2 if _is_pair else None, age=e.age.values, women=(~e.is_male.astype(bool)).values,
                               labels=None if _is_pair else q[["NORM", "CD"]]))

# EchoNext
q = eval_records[eval_records.set == "echonext_test"]; e = echo.iloc[q.row.values]
_lines.append(describe_set("echonext_test", "echonext", len(q), q.patient.nunique(), age=pd.to_numeric(e.age_at_ecg, errors="coerce").values, women=(e.sex.astype(str).str.lower() == "female").values, labels=q[["HYP"]]))

characteristics = pd.DataFrame(_lines).astype({"pairs": "Int64", "interval_within_1h_n": "Int64"})
characteristics.to_csv(SET_CHARACTERISTICS, index=False)
print(characteristics.round(2).to_string(index=False))

## File formats

All files are below `MASTER/data/derived/`. The formats of `sets/ptbxl_sets.csv`, `sets/mimic_pool_records.csv` and `sets/eval_records.csv` are used by notebooks 03–08.

**`sets/ptbxl_sets.csv`** (`PTBXL_SETS`): one line per recording of PTB-XL (21,799).

| Column | Content |
|---|---|
| `row` | row of `ptbxl_filtered_f16.npy` (= row of `ptbxl_database.csv`) |
| `ecg_id`, `patient`, `fold` | `ecg_id`, `patient_id` and `strat_fold` of PTB-XL |
| `set` | `train` (folds 1–8), `val` (fold 9), `test` (fold 10) or `excluded` (no diagnostic statement) |
| `adjacent_ok` | 1 = test recording that is used for the comparison of adjacent segments |
| `NORM`, `MI`, `STTC`, `CD`, `HYP` | reference labels (0/1) |

**`sets/mimic_pool_records.csv`** (`POOL_RECORDS`): one line per recording of the unlabelled training pool: `patient` (`subject_id`), `record` (`study_id`), `row` (row of `mimic_pool_f16.npy`).

**`sets/eval_records.csv`** (`EVAL_RECORDS`): one line per recording and evaluation set; the order of the lines is the order of all model outputs.

| Column | Content |
|---|---|
| `dataset` | `ptbxl`, `mimic`, `code15` or `echonext` |
| `set` | `ptbxl_val`, `ptbxl_test`, `mimic_test`, `mimic_pairs_all`, `mimic_pairs_same_report`, `code15_test`, `code15_cd_change`, `code15_normal_change`, `echonext_test` |
| `patient` | patient identifier of the dataset (`patient_id`, `subject_id`, `patient_id`, `patient_key`) |
| `record` | recording identifier: `ecg_id` (PTB-XL), `study_id` (MIMIC-IV-ECG), `exam_id` (CODE-15%), row of the cached array (EchoNext) |
| `pair_id` | number of the pair within the set; −1 for single recordings |
| `role` | `single`; `a` and `b` = earlier and later recording of a MIMIC-IV-ECG pair; `pos` and `neg` = recording with and without the label of a CODE-15% pair |
| `array`, `row` | cached array (`ptbxl`, `mimic_eval`, `code15/part{i}`, `echonext`) and row in it |
| `interval_sec` | interval between the two recordings of a MIMIC-IV-ECG pair, in seconds (empty otherwise) |
| `adjacent_ok` | 1 = used for the comparison of adjacent segments |
| `NORM`, `MI`, `STTC`, `CD`, `HYP` | reference labels (empty where the dataset has none; EchoNext: `HYP` = wall thickness of 13 mm or more) |

**`mimic_waveform_check.csv`** (`MIMIC_CHECK`): one line per recording of MIMIC-IV-ECG that was examined:
`record` (`study_id`), `readable` (0 = the file could not be read or the signal was zero everywhere), `n_r_first`, `n_r_second` (R waves detected in the first and in the second 5-s half of lead II), `md5` (checksum of the waveform file).
A recording passes when `readable` = 1 and both counts are at least 3.

**`sets/set_counts.json`** (`SET_COUNTS`): numbers of the flow diagram.

```
ptbxl:    recordings, patients, excluded_no_diagnostic_statement, recordings_with_diagnostic_statement, train, val, test,
          test_excluded_by_statement, test_failed_waveform_check, test_adjacent
mimic:    recordings, patients, excluded_by_statement, excluded_by_statement_reasons {reason: recordings; a recording can have several reasons},
          recordings_after_statement_exclusion, patients_after_statement_exclusion, recordings_time_mismatch (all recordings whose two times differ),
          pool {patients_drawn, recordings_examined, recordings_failed_waveform_check, recordings, patients},
          evaluation_candidate_patients, pair_candidate_patients_before_time_exclusion,
          recordings_excluded_from_pairs_time_mismatch (recordings of the candidates for evaluation whose two times differ),
          sets {mimic_test, mimic_pairs_all, mimic_pairs_same_report}
code15:   exams, patients, exams_cached, excluded_shorter_than_10_s, exams_after_exclusion, patients_after_exclusion,
          sets {code15_cd_change, code15_normal_change, code15_test}
echonext: recordings, patients, sets {echonext_test}
each set: candidates (patients after the exclusions), examined (patients, up to the one who completed the set), selected,
          rejected {reason: patients}, recordings_failed_waveform_check, recordings_dropped_identical_file (among the patients examined)
```

Reasons for rejection: `waveform_check` (no recording, or no pair, was left after the waveform check), `identical_file` (no pair was left after a recording with an identical waveform file was dropped),
`waveform_check_and_identical_file`.

**`sets/set_characteristics.csv`** (`SET_CHARACTERISTICS`): one line per set, in the order of Supplementary Table 1
(`ptbxl_train`, `ptbxl_val`, `ptbxl_test`, `ptbxl_test_adjacent`, `mimic_pool`, `mimic_test`, `mimic_pairs_all`, `mimic_pairs_same_report`, `code15_test`, `code15_cd_change`, `code15_normal_change`, `echonext_test`).

| Column | Content |
|---|---|
| `set`, `dataset` | name of the set and of the dataset |
| `recordings`, `patients` | numbers of recordings and of patients |
| `pairs` | number of pairs (empty for sets of single recordings) |
| `age_n`, `age_median`, `age_q1`, `age_q3` | recordings with a known age; median and quartiles of the age in years, over recordings, ages above 89 set to 90 |
| `sex_n`, `women_n`, `women_pct` | recordings with a known sex; recordings of women; percentage of women (100 × `women_n` / `sex_n`) |
| `prevalence_NORM` … `prevalence_HYP` | percentage of recordings with the reference label (empty where the dataset has no such label, and for sets of pairs) |
| `interval_h_median`, `interval_h_q1`, `interval_h_q3` | MIMIC-IV-ECG pairs: median and quartiles of the interval in hours |
| `interval_within_1h_n`, `interval_within_1h_pct` | MIMIC-IV-ECG pairs: number and percentage of pairs with an interval of at most 1 h |

Quartiles are calculated with linear interpolation (`np.percentile`). Percentages are written without rounding.
For a percentage that lies exactly halfway between two printed values, the printed digit depends on the rounding rule and on how the file is read:
`pd.read_csv(SET_CHARACTERISTICS, float_precision="round_trip")` returns exactly the stored numbers, and the counts (`women_n`, `sex_n`, `interval_within_1h_n`, `pairs`) allow exact arithmetic.